[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week0_model_is_broken_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 0 -- The Model Is Broken (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** MLOps stack setup, clone the broken model, reproduce the failure, map the production environment

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Clone a broken production ML pipeline and reproduce the failure from git history
- Map the full production stack: data sources, preprocessing, model serving, monitoring
- Distinguish pipeline failure (system broke) from model failure (wrong predictions)
- Set up the MLOps environment: DVC, MLflow, FastAPI, Docker, Evidently
- Write a one-page failure summary stating what broke, when, and what is unknown



---

## MONDAY -- "Clone, Run, Fail"


Your first task is not to train a model. It is to make someone else's model run on your machine. If you cannot reproduce the production environment locally, you cannot debug it. Clone the repo, set up the environment, run the model. It will fail. Document the failure.


### Exercise 0.1 -- Clone and reproduce

Clone the repo. Run serve.py. Document: the exact error, the line that fails, what the error tells you about the missing dependency.


In [ ]:
# Exercise 0.1: Clone and reproduce
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


### Exercise 0.2 -- Environment audit

Compare requirements.txt with the production pip freeze. List every discrepancy.


In [ ]:
# Exercise 0.2: Environment audit
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Clone, Run, Fail --
git clone https://github.com/dataflow-internal/credit-default-v2.git
cd credit-default-v2
python -m venv venv && source venv/bin/activate     # isolate this project's dependencies
pip install -r requirements.txt
python serve.py  # document the exact error -- do not paraphrase it in your notes


### Expected Output (illustrative)

```
$ python serve.py
Traceback (most recent call last):
  File "serve.py", line 14, in <module>
    model = joblib.load("models/credit_default_v2.pkl")
  File "/venv/lib/python3.10/site-packages/joblib/numpy_pickle.py", line 650, in load
    obj = _unpickle(fobj, filename, mmap_mode)
sklearn.exceptions.InconsistentVersionWarning: Trying to unpickle estimator
RandomForestClassifier from version 1.2.2 when using version 1.4.2.
```
The server *starts*. The health check may even return `200`. This is the trap: an environment
mismatch like this can produce silently different predictions rather than a hard crash --
exactly the "clean logs, wrong decisions" failure mode this week is about.


### Monday Morning Moment

*DataFlow Infrastructure -- Monday, 9:30am.*

**Sola Fashola:** You cloned the repo?

**Temi Adeyemi:** Yes. Fails with ModuleNotFoundError on featuretools.

**Sola Fashola:** featuretools was removed in the July update. Production server still has the old version pinned. This is why we version environments.

**Temi Adeyemi:** So local and production are different.

**Dr. Emeka Obi:** Write that down. That is your first finding.




---

## TUESDAY -- "Map the Production Stack"


Before fixing a broken pipeline, draw it. Every data source, every transformation, every output. A stack map takes 30 minutes and saves hours of debugging.


### Exercise 0.3 -- Stack map

Draw the full production stack. For each arrow: one diagnostic check you could run to verify that connection is working.


In [ ]:
# Exercise 0.3: Stack map
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Map the Production Stack --
# Stack diagram -- mark each arrow with a diagnostic check:
# [Cooperative DB] -> [preprocess.py] -> [credit_default_v2.pkl] -> [FastAPI /predict] -> [Retool dashboard]
# Where could it break? What check tests each link?


### Expected Output (illustrative)

There's no code to run here — Tuesday's deliverable is the stack diagram itself, annotated with
one diagnostic check per arrow, e.g.:

```
[Cooperative DB] --(schema check: 23 cols present?)--> [preprocess.py]
[preprocess.py]  --(null/range check on output?)------> [credit_default_v2.pkl]
[credit_default_v2.pkl] --(model version pinned?)-----> [FastAPI /predict]
[FastAPI /predict] --(latency + status code?)---------> [Retool dashboard]
```
The point of the exercise: a healthy dashboard tells you the *last* arrow works. It tells you
nothing about the first three.



---

## WEDNESDAY -- "Pipeline Failure vs Model Failure"


Different problems, different fixes. Pipeline failure: infrastructure broke. Model failure: predictions are wrong for the data being seen. Do not assume which before you have evidence.

Pause and Predict -- before running any diagnostic commands: based on the symptoms so far (gradual approval rate increase, no errors in logs, model weights unchanged), which is more likely -- pipeline failure, model failure, or data drift? Write your prediction before running the commands below.


### Exercise 0.4 -- Failure classification

Given the symptoms, classify as pipeline failure, model failure, or data drift. Justify with two pieces of evidence.


In [ ]:
# Exercise 0.4: Failure classification
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Pipeline Failure vs Model Failure --
# Check logs first:
import subprocess
result = subprocess.run(["cat", "logs/serve.log"], capture_output=True, text=True)
print(result.stdout[-3000:])          # tail the log -- most recent errors are what matter
# Pipeline failure: exceptions, NaN outputs, HTTP 500s
# Model failure: clean logs, plausible-looking predictions, wrong decisions


### Expected Output (illustrative)

```
2026-06-08 09:14:02 INFO  200 POST /predict {"customer_id": 18442, "decision": "APPROVE"}
2026-06-08 09:14:03 INFO  200 POST /predict {"customer_id": 18443, "decision": "APPROVE"}
2026-06-08 09:14:04 INFO  200 POST /predict {"customer_id": 18444, "decision": "APPROVE"}
```
No exceptions. No 500s. Every request returns `200`. **This is a model failure, not a
pipeline failure** — the logs are clean because nothing is technically broken. The bug is in
what the model is deciding, not in whether it runs.



---

## THURSDAY -- "The MLOps Stack"


Install and verify the full MLOps stack: DVC, MLflow, Airflow, Evidently, Prometheus. You will use every one before Week 12.


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: The MLOps Stack --
pip install dvc mlflow apache-airflow evidently prometheus-client
import mlflow
print(f"MLflow: {mlflow.__version__}")
import evidently
print(f"Evidently: {evidently.__version__}")


### Expected Output (illustrative)

```
MLflow: 2.12.0
Evidently: 0.4.25
```
Exact versions will match whatever `requirements.txt` pins at install time (`mlflow>=2.12.0`,
`evidently>=0.4.0`) — pin these in your own `requirements.txt` too, since an MLOps stack that
silently upgrades itself is the same class of problem as Monday's environment mismatch.



---

## FRIDAY -- "The Friday Build: Failure Summary"


Write the failure summary for Dr. Emeka. One page. Status, impact, what we know, what we suspect, what we ruled out, next actions, risk if unresolved.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Failure Summary --
# STATUS: RED
# IMPACT: Approval rate +12pp since Q1.
# WHAT WE KNOW: [3-5 confirmed facts]
# WHAT WE SUSPECT: [2-3 hypotheses]
# WHAT WE RULED OUT: [cleared checks]
# NEXT ACTIONS: [3 items with owner and due date]
# RISK IF UNRESOLVED: [one sentence for Farida]


### Expected Output (illustrative)

The Friday Build has no console output — it's a written failure summary. A STATUS: RED
summary that would satisfy Dr. Emeka looks like:

```
STATUS: RED
IMPACT: Approval rate +12pp since Q1; ~1,840 Q4 decisions at elevated risk.
WHAT WE KNOW: server healthy; logs clean; predictions systematically shifted toward APPROVE.
WHAT WE SUSPECT: population or feature drift in the credit-limit distribution.
WHAT WE RULED OUT: infrastructure failure, environment/version mismatch.
NEXT ACTIONS: (1) run PSI/KS on Q1 vs Q4 [owner: you, due Mon], (2) SHAP comparison [owner: you, due Tue].
RISK IF UNRESOLVED: approval decisions continue drifting with no monitoring to catch it.
```


### Friday Workplace Moment

*DataFlow Infrastructure -- Friday standup.*

**Dr. Emeka Obi:** Failure summary. What do we know?

**Temi Adeyemi:** Model weights unchanged since January. Serving shows no errors in logs. Approval rate increased gradually from February -- not suddenly.

**Dr. Emeka Obi:** Gradual. Rules out a deployment event. Hypothesis?

**Temi Adeyemi:** Loan amount distribution shifted -- larger loans, different risk profiles.

**Farida Usman:** Is the model safe to use?

**Temi Adeyemi:** Not confidently for large loans. I recommend flagging approvals above a threshold for manual review until we understand the drift.

**Dr. Emeka Obi:** Write that mitigation into the failure summary.



## YOUR WEEK 0 CHECKLIST

Check each item before moving on.

- [ ] I can clone the repository and reproduce the environment error from first principles.
- [ ] I have drawn the full production stack with every failure point identified.
- [ ] I can distinguish pipeline failure from model failure with evidence, not assumption.
- [ ] The MLOps stack is installed and verified on my machine.
- [ ] My failure summary leads with impact and risk, not with methodology.


## Challenge Task

> **Core path:** Write a Docker Compose file reproducing the production environment exactly, including the pinned featuretools version.

> **Advanced path:** Use git bisect to identify the exact commit where requirements.txt and the production pip freeze diverged.


## Common Mistakes

**Assuming the model is broken when the pipeline is broken:** 80% of production ML failures are infrastructure. Check logs and model weights before touching model code.

**Trying to fix before understanding:** The failure summary this week is a diagnosis. A wrong fix applied confidently is worse than no fix.

**Treating environment discrepancy as minor:** A mismatch between requirements.txt and production is a ticking clock. The server will eventually restart.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)